In [15]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Mizard\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Mizard\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Mizard\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [14]:
import pandas as pd
import numpy as np
import re
import seaborn as sns
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
from nltk.stem import PorterStemmer
from langdetect import detect, LangDetectException

In [10]:
df = pd.read_csv('threads_reviews.csv')
print("="*35)

print(df.head().to_markdown(index=False, numalign="left", stralign="left"))
print("="*35)

print(df.info())
print("="*35)

print(df['rating'].unique())
print("="*35)

display(df)

| source      | review_description                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                | rating   | review_date         |
|:------------|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

,source,review_description,rating,review_date
0,Google Play,Meh. Not the greatest experience on a Chromebo...,2,2023-07-08 14:18:24
1,Google Play,Pretty good for a first launch!! Its easy to u...,3,2023-07-19 20:52:48
2,Google Play,"For a brand new app, it's very well optimized....",3,2023-07-06 23:03:11
3,Google Play,"Great app with a lot of potential! However, th...",3,2023-07-10 00:53:25
4,Google Play,"The app is good, but it needs a lot of functio...",3,2023-07-06 16:57:43
...,...,...,...,...
32905,App Store,This killed my dog. Mark zuckerburg strangled ...,1,2023-07-06 01:23:55
32906,App Store,Add Search and hashtag like Twitter !,1,2023-07-19 08:01:06
32907,App Store,bad twister,1,2023-07-17 06:39:13
32908,App Store,Yet another trash from Meta.,1,2023-07-07 17:47:16


In [ ]:
def get_sentiment(rating):
    if rating <= 2:
        return 'Negative'
    elif rating == 3:
        return 'Neutral'
    else:
        return 'Positive'

df['sentiment'] = df['rating'].apply(get_sentiment)

negative_reviews = df[df['sentiment'] == 'Negative']
neutral_reviews = df[df['sentiment'] == 'Neutral']
positive_reviews = df[df['sentiment'] == 'Positive']

sampled_positive = positive_reviews.sample(n=100, random_state=42)
sampled_negative = negative_reviews.sample(n=100, random_state=42)
sampled_neutral = neutral_reviews.sample(n=50, random_state=42)

final_df_500 = pd.concat([sampled_positive, sampled_negative, sampled_neutral])
final_df_500 = final_df_500.sample(frac=1, random_state=42).reset_index(drop=True)
final_df_500.to_csv('sampled_threads_reviews.csv', index=False)
print("="*35)

print(final_df_500['sentiment'].value_counts())
print("="*35)
print(final_df_500.info())

sentiment
Negative    100
Positive    100
Neutral      50
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   source              250 non-null    object
 1   review_description  250 non-null    object
 2   rating              250 non-null    int64 
 3   review_date         250 non-null    object
 4   sentiment           250 non-null    object
dtypes: int64(1), object(4)
memory usage: 9.9+ KB
None


In [12]:
data = final_df_500['review_description'].apply(sent_tokenize)

total_documents = len(data)
print(f"\nTotal dokumen yang ditokenisasi: {total_documents}")
print(data.head())


Total dokumen yang ditokenisasi: 250
0                                               [چرته]
1    [Not perfect but could be great if they begin ...
2                                          [Excellent]
3                                    [The app is good]
4                      [Issue it fails to install!, !]
Name: review_description, dtype: object


In [ ]:
stemmer = PorterStemmer()
stop_words = stopwords.words('english')

def preprocess(text):
  text = re.sub(r'<[^>]+>|https?://\S+|[^a-zA-Z0-9\s]', ' ', text.lower())
  tokens = text.split()
  filtered = [w for w in tokens if w not in stop_words]
  stemmed = [stemmer.stem(word) for word in filtered]
  return ' '.join(stemmed)
print(final_df_500[['review_description', 'sentiment']].head())

for i in range(1, 6):
    print(f"\n--- Sampel Rating {i} ---")
    sampel = final_df_500[final_df_500['rating'] == i].head(2)
    print(sampel[['review_description', 'sentiment']])

                                  review_description sentiment
0                                               چرته  Negative
1  Not perfect but could be great if they begin m...  Positive
2                                          Excellent  Positive
3                                    The app is good  Positive
4                        Issue it fails to install!!  Negative

--- Sampel Rating 1 ---
            review_description sentiment
0                         چرته  Negative
4  Issue it fails to install!!  Negative

--- Sampel Rating 2 ---
                                  review_description sentiment
7  Maganda siya, problema lang is wala akong ka p...  Negative
9  What going on Thread! I can type but I can't s...  Negative

--- Sampel Rating 3 ---
                                   review_description sentiment
10  Connecting and redirecting to Instagram resets...   Neutral
11  Hey ,please don't make same color and options ...   Neutral

--- Sampel Rating 4 ---
   review_descript

In [ ]:
# --- BAGIAN 1: FUNGSI FILTER BAHASA ---
def is_english(text):
    try:
        return detect(text) == 'en'
    except LangDetectException:
        return False

# --- BAGIAN 2: LOAD & FILTER DATA ---
df = pd.read_csv('threads_reviews.csv')
print("Jumlah data awal:", len(df))

print("Sedang memfilter bahasa Inggris (mohon tunggu)...")
df = df[df['review_description'].apply(is_english)]
print("Jumlah data setelah filter bahasa Inggris:", len(df))

# --- BAGIAN 3: SAMPLING BERDASARKAN SENTIMEN ---
def get_sentiment(rating):
    if rating <= 2:
        return 'Negative'
    elif rating == 3:
        return 'Neutral'
    else:
        return 'Positive'

df['sentiment'] = df['rating'].apply(get_sentiment)

# Separate by sentiment
negative_reviews = df[df['sentiment'] == 'Negative']
neutral_reviews = df[df['sentiment'] == 'Neutral']
positive_reviews = df[df['sentiment'] == 'Positive']

try:
    sampled_positive = positive_reviews.sample(n=200, random_state=42)
    sampled_negative = negative_reviews.sample(n=200, random_state=42)
    sampled_neutral = neutral_reviews.sample(n=100, random_state=42)
except ValueError:
    print("Peringatan: Data bahasa Inggris tidak cukup untuk memenuhi target kuota.")
    sampled_positive = positive_reviews.head(200)
    sampled_negative = negative_reviews.head(200)
    sampled_neutral = neutral_reviews.head(100)

# Concatenate & Shuffle
final_df = pd.concat([sampled_positive, sampled_negative, sampled_neutral])
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)
final_df.to_csv('sampled_threads_reviews.csv', index=False)

# --- BAGIAN 4: TOKENISASI ---
print("\n--- Proses Tokenisasi ---")
data_tokenized = final_df['review_description'].apply(sent_tokenize)

print(f"Total dokumen final: {len(data_tokenized)}")
print(final_df['sentiment'].value_counts())

print("\nContoh data (Bahasa Inggris):")
print(final_df[['review_description', 'sentiment']].head())

Jumlah data awal: 32910
Sedang memfilter bahasa Inggris (mohon tunggu)...
Jumlah data setelah filter bahasa Inggris: 21768

--- Proses Tokenisasi ---
Total dokumen final: 500
sentiment
Negative    200
Positive    200
Neutral     100
Name: count, dtype: int64

Contoh data (Bahasa Inggris):
                                  review_description sentiment
0  I’m excited for this app but the MAJORITY of m...  Negative
1  First downloader so follow me Insta id. sam_as...  Positive
2                                     Not bad at all  Negative
3                               Needed some new app.  Positive
4  Haven't used the app yet but from what I see f...  Positive


In [5]:
# Load datasets
try:
    data_manual = pd.read_csv('data_manual.csv')
    full_data = pd.read_csv('threads_reviews.csv')
    print("Data Manual Loaded. Shape:", data_manual.shape)
    print("Full Data Loaded. Shape:", full_data.shape)
    print("Data Manual Columns:", data_manual.columns)
except Exception as e:
    print("Error loading data:", e)

Data Manual Loaded. Shape: (500, 5)
Full Data Loaded. Shape: (32910, 4)
Data Manual Columns: Index(['source', 'review_description', 'rating', 'review_date', 'sentiment'], dtype='object')


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC  
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

try:
    from langdetect import detect
except ImportError:
    pass

# 1. Load Data
try:
    data_manual = pd.read_csv('data_manual.csv')
    full_data = pd.read_csv('threads_reviews.csv')
except FileNotFoundError:
    print("Error: File CSV tidak ditemukan. Pastikan file ada di direktori yang sama.")
    exit()

# 2. Identify "Remaining" Data (Data in full_data NOT in data_manual)
common_cols = ['source', 'review_description', 'rating', 'review_date']

# Ensure data types match for merging
for col in common_cols:
    if col in data_manual.columns and col in full_data.columns:
        data_manual[col] = data_manual[col].astype(str)
        full_data[col] = full_data[col].astype(str)

merged = full_data.merge(data_manual[common_cols], on=common_cols, how='left', indicator=True)
remaining_data = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge'])

print(f"Total Original Data: {len(full_data)}")
print(f"Data in Manual Set: {len(data_manual)}")
print(f"Remaining Data Pool: {len(remaining_data)}")

# 3. Sample 750 data points from the remaining pool
remaining_data = remaining_data.sample(frac=1, random_state=42).reset_index(drop=True)

try:
    def is_english(text):
        try:
            return detect(text) == 'en'
        except:
            return False

    print("Mencoba memfilter 750 ulasan berbahasa Inggris...")
    english_samples = []
    count = 0

    for index, row in remaining_data.iterrows():
        if is_english(row['review_description']):
            english_samples.append(row)
            count += 1
        if count >= 750:
            break

    if count < 750:
        print(f"Peringatan: Hanya menemukan {count} ulasan Inggris. Menggunakan seadanya.")

    sample_750 = pd.DataFrame(english_samples)
    print(f"Berhasil mengambil sampel: {len(sample_750)}")

except (ImportError, NameError) as e:
    print(f"Langdetect tidak tersedia atau error ({e}). Mengambil 750 sampel acak tanpa filter bahasa.")
    sample_750 = remaining_data.head(750)
except Exception as e:
    print(f"Terjadi kesalahan deteksi bahasa ({e}). Mengambil 750 sampel acak.")
    sample_750 = remaining_data.head(750)

if sample_750.empty:
    print("Tidak ada data untuk diprediksi.")
    exit()

# 4. Prepare Training Data
X_train = data_manual['review_description']
y_train = data_manual['sentiment']

# 5. Feature Extraction (TF-IDF)
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)

# 6. Train Model (MENGGUNAKAN SVM)
svm_model = SVC(kernel='linear', random_state=42)
print("Sedang melatih model SVM...")
svm_model.fit(X_train_vec, y_train)

# 7. Predict on Sampled Data
X_test = sample_750['review_description']
X_test_vec = vectorizer.transform(X_test)
predicted_labels = svm_model.predict(X_test_vec)

# 8. Add predictions to the dataframe
sample_750['predicted_sentiment'] = predicted_labels

# 9. Save to CSV
output_filename = 'label_review_SVM.csv'
sample_750.to_csv(output_filename, index=False)

# Display results
print(f"\nPrediksi selesai! Disimpan ke '{output_filename}'")
print("\nDistribusi Prediksi:")
print(sample_750['predicted_sentiment'].value_counts())
print("\nContoh Hasil Prediksi:")

Total Original Data: 32910
Data in Manual Set: 500
Remaining Data Pool: 32410
Mencoba memfilter 750 ulasan berbahasa Inggris...
Berhasil mengambil sampel: 750
Sedang melatih model SVM...

Prediksi selesai! Disimpan ke 'labeled_750_reviews_svm.csv'

Distribusi Prediksi:
predicted_sentiment
Negative    355
Positive    348
Neutral      47
Name: count, dtype: int64

Contoh Hasil Prediksi:
                                  review_description predicted_sentiment
0  The IG & Thread integration is top tier, excit...            Positive
2  App is good make more flexible like stories sh...            Positive
3                        Best Compitator for Twitter            Positive
4                                          Nicee app            Negative
6  The old threads app was way more useful than t...            Positive
